# MT5 Portfolio Analyzer

Runs the four-pair grid strategy portfolio reconstruction from MT5 backtest exports and displays all key results interactively.

**What this notebook does:**
1. Runs the simulation engine against your XLSX + tester-graph CSV files
2. Displays a combined balance & equity curve over time
3. Shows drawdown profile
4. Shows per-pair cumulative PnL contribution
5. Displays a summary metrics table

**Data directory:** `data/`  
**Config:** `config/example_config.json`

In [1]:
import sys, os
REPO = os.path.dirname(os.path.abspath("portfolio_analyzer.ipynb"))
SRC  = os.path.join(REPO, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import warnings
warnings.filterwarnings("ignore")

print(f"Repo root : {REPO}")
print(f"Python    : {sys.executable}")

import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# Import analyzer types and functions
from mt5_portfolio_analyzer import (
    DealEvent, TradeEvent, PairConfig, PairData,
    run_simulation, _interpolate_floating,
)

# nbformat was installed after the kernel started — re-inject it so plotly sees it
import importlib
import nbformat
import plotly.io._renderers as _plotly_renderers
_plotly_renderers.nbformat = nbformat

pio.renderers.default = "notebook_connected"

print("pandas", pd.__version__, "| plotly", plotly.__version__,
      "| nbformat", nbformat.__version__, "| renderer:", pio.renderers.default)

Repo root : c:\Users\jsotoalv\source\repos\mt5-portfolio-analyzer
Python    : c:\Users\jsotoalv\source\repos\mt5-portfolio-analyzer\.venv\Scripts\python.exe
pandas 3.0.3 | plotly 6.8.0 | nbformat 5.10.4 | renderer: notebook_connected


## Step 1 — Run the Simulation

Loads all four XLSX reports and tester-graph CSVs from `data/`, runs the combined portfolio simulation, and stores the results in memory.

In [2]:
from IPython.display import display
import importlib
import numpy as np
import mt5_readers as mtr
import mt5_portfolio_analyzer as mpa

# Ensure notebook picks up latest reader/engine edits without a manual kernel restart.
importlib.reload(mtr)
importlib.reload(mpa)

from mt5_readers import discover_files, load_xlsx_deals
from mt5_portfolio_analyzer import (
    load_pair, run_simulation,
    _PAIR_RISK, _PAIR_BASE_LOT,
)

# Use baseline scenario (update to 'proposed' to run proposed scenario)
DATA_DIR = os.path.join(REPO, "data", "baseline")

# --- Simulation parameters (edit as needed) ---
INITIAL_BALANCE = 5000.0
SCALE_EXPONENT  = 1.0
MIN_SCALE       = 0.1
MAX_SCALE       = 5.0

# --- Broker margin requirements ---
MMR_CSV = os.path.join(REPO, "data", "reference", "forex_com_margin_requirements.csv")
margin_requirements = {}
if os.path.exists(MMR_CSV):
    mmr_df = pd.read_csv(MMR_CSV)
    for _, row in mmr_df.iterrows():
        pair_key = "".join(ch for ch in str(row["currency_pair"]).upper() if ch.isalpha())
        margin_requirements[pair_key] = float(row["mmr_percent"])
    print(f"Loaded MMR for {len(margin_requirements)} pairs from {MMR_CSV}")
else:
    print("MMR file not found, margin metrics will be blank.")


def average_monthly_balance_growth(curve_rows):
    df = pd.DataFrame(curve_rows)
    if df.empty:
        return None
    df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M")
    df["ym"] = df["time"].dt.to_period("M")
    monthly_bal = df.groupby("ym")["balance"].last()
    if len(monthly_bal) < 2:
        return None
    returns = monthly_bal.pct_change().dropna()
    return returns.mean() * 100 if len(returns) > 0 else None


def recompute_shared_dd(curve_rows):
    """Recompute DD on shared portfolio capital from equity curve."""
    df = pd.DataFrame(curve_rows)
    if df.empty or "equity" not in df.columns:
        return {
            "max_drawdown_abs": 0.0,
            "max_drawdown_percent": 0.0,
            "max_drawdown_time": None,
            "max_drawdown_peak_equity": None,
            "equity_at_max_drawdown": None,
        }

    if "time" in df.columns:
        df = df.copy()
        df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M", errors="coerce")
        df = df.sort_values("time").reset_index(drop=True)

    equity = pd.to_numeric(df["equity"], errors="coerce").fillna(0.0).to_numpy()
    if len(equity) == 0:
        return {
            "max_drawdown_abs": 0.0,
            "max_drawdown_percent": 0.0,
            "max_drawdown_time": None,
            "max_drawdown_peak_equity": None,
            "equity_at_max_drawdown": None,
        }

    peak = pd.Series(equity).cummax().to_numpy()
    dd_abs = peak - equity
    dd_pct = np.where(peak > 0, dd_abs / peak * 100.0, 0.0)

    dd_idx = int(np.argmax(dd_pct))
    dd_time = None
    if "time" in df.columns and dd_idx < len(df):
        t = df.iloc[dd_idx]["time"]
        dd_time = t.strftime("%Y.%m.%d %H:%M") if pd.notna(t) else None

    return {
        "max_drawdown_abs": float(dd_abs[dd_idx]),
        "max_drawdown_percent": float(dd_pct[dd_idx]),
        "max_drawdown_time": dd_time,
        "max_drawdown_peak_equity": float(peak[dd_idx]),
        "equity_at_max_drawdown": float(equity[dd_idx]),
    }


def apply_shared_dd(summary_dict, curve_rows):
    out = dict(summary_dict) if summary_dict is not None else {}
    out.update(recompute_shared_dd(curve_rows))
    return out


# --- Load all pairs ---
discovered = discover_files(DATA_DIR)
pairs_data = []
for pair, files in discovered.items():
    xlsx = files.get("xlsx")
    csv  = files.get("csv")
    if not xlsx:
        print(f"  SKIP {pair}: no xlsx found")
        continue
    print(f"  Loading {pair} ...")
    pairs_data.append(load_pair(
        name=pair,
        risk_percent=_PAIR_RISK.get(pair, 0.0),
        base_lot=_PAIR_BASE_LOT.get(pair),
        xlsx_path=xlsx,
        csv_path=csv,
    ))

# --- Run combined simulation ---
result = run_simulation(
    pairs_data=pairs_data,
    initial_balance=INITIAL_BALANCE,
    scale_exponent=SCALE_EXPONENT,
    min_scale=MIN_SCALE,
    max_scale=MAX_SCALE,
    margin_requirements=margin_requirements,
)

summary = apply_shared_dd(result["summary"], result["curve_rows"])
result["summary"] = summary
baseline_avg_monthly_growth = average_monthly_balance_growth(result["curve_rows"])

print(f"\nDone. {summary['total_deals']} deals | "
      f"{summary['period_start']} -> {summary['period_end']}")
if baseline_avg_monthly_growth is not None:
    print(f"Avg Monthly Balance Growth: {baseline_avg_monthly_growth:.3f}%")

Loaded MMR for 80 pairs from c:\Users\jsotoalv\source\repos\mt5-portfolio-analyzer\data\reference\forex_com_margin_requirements.csv
  Loading EURUSD ...
  Loading EURGBP ...
  Loading GBPUSD ...
  Loading USDCHF ...

Done. 2461 deals | 2023.01.02 00:00:00 -> 2026.06.04 23:59:58
Avg Monthly Balance Growth: 8.536%


## Step 2 — Summary Metrics Table

In [3]:
from IPython.display import display

if "baseline_avg_monthly_growth" not in globals():
    baseline_avg_monthly_growth = average_monthly_balance_growth(result["curve_rows"])

# --- Portfolio-level summary ---
portfolio_metrics = {
    "Metric": [
        "Initial Balance",
        "Final Balance",
        "Total Return",
        "CAGR",
        "Avg Monthly Balance Growth",
        "Max Drawdown (abs)",
        "Max Drawdown (%)",
        "Max Used Margin",
        "Min Free Margin",
        "Min Margin Level (%)",
        "Total Deals",
        "Period",
    ],
    "Value": [
        f"${summary['initial_balance']:,.2f}",
        f"${summary['final_balance']:,.2f}",
        f"{summary['total_return_percent']:.2f}%",
        f"{summary['cagr_percent']:.2f}%",
        f"{baseline_avg_monthly_growth:.3f}%" if baseline_avg_monthly_growth is not None else "N/A",
        f"${summary['max_drawdown_abs']:,.2f}",
        f"{summary['max_drawdown_percent']:.2f}%",
        f"${summary['max_used_margin']:,.2f}" if summary.get('max_used_margin') is not None else "N/A",
        f"${summary['min_free_margin']:,.2f}" if summary.get('min_free_margin') is not None else "N/A",
        f"{summary['min_margin_level_percent']:.2f}%" if summary.get('min_margin_level_percent') is not None else "N/A",
        str(summary["total_deals"]),
        f"{summary['period_start']}  ->  {summary['period_end']}",
    ],
}
df_summary = pd.DataFrame(portfolio_metrics)
display(df_summary.style.hide(axis="index").set_caption("Portfolio Summary"))

# --- Per-pair breakdown ---
pair_rows = []
for pair, info in summary["pairs"].items():
    pair_rows.append({
        "Pair":            pair,
        "Risk %":          info["risk_percent"],
        "Deals":           info["deals_count"],
        "Median Vol":      info["baseline_volume_median"],
        "Scaled PnL ($)":  f"{info['scaled_pnl_contribution']:+,.2f}",
    })
df_pairs = pd.DataFrame(pair_rows)
display(df_pairs.style.hide(axis="index").set_caption("Per-Pair Contribution"))

Metric,Value
Initial Balance,"$5,000.00"
Final Balance,"$154,949.70"
Total Return,2998.99%
CAGR,172.73%
Avg Monthly Balance Growth,8.536%
Max Drawdown (abs),"$17,887.00"
Max Drawdown (%),134.83%
Max Used Margin,"$52,169.64"
Min Free Margin,"$-26,690.42"
Min Margin Level (%),-20.94%


Pair,Risk %,Deals,Median Vol,Scaled PnL ($)
EURUSD,3.300000,336,0.350000,"+86,010.61"
EURGBP,1.200000,601,0.070000,"+9,739.75"
GBPUSD,2.000000,446,0.150000,"+26,954.94"
USDCHF,1.500000,1078,0.110000,"+27,244.39"


## Step 3 — Combined Balance, Equity & Drawdown

In [12]:
df_curve = pd.DataFrame(result["curve_rows"])
df_curve["time"] = pd.to_datetime(df_curve["time"], format="%Y.%m.%d %H:%M")

equity = df_curve["equity"].values
peak = pd.Series(equity).cummax()
dd_abs = peak - equity
dd_pct = (dd_abs / peak * 100).where(peak > 0, 0)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.52, 0.24, 0.24],
    subplot_titles=["Combined Balance & Equity", "Drawdown (USD)", "Drawdown (%)"],
    vertical_spacing=0.06,
)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=df_curve["balance"],
    name="Balance", line=dict(color="#2196F3", width=1.6),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=df_curve["equity"],
    name="Equity", line=dict(color="#4CAF50", width=1.2, dash="dot"),
    fill="tonexty", fillcolor="rgba(76,175,80,0.08)",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=-dd_abs,
    fill="tozeroy", fillcolor="rgba(244,67,54,0.20)",
    line=dict(color="#F44336", width=1),
    name="DD abs ($)",
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=-dd_pct,
    fill="tozeroy", fillcolor="rgba(244,67,54,0.15)",
    line=dict(color="#E91E63", width=1),
    name="DD pct (%)",
), row=3, col=1)

fig.add_hline(
    y=INITIAL_BALANCE,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Start ${INITIAL_BALANCE:,.0f}",
    row=1,
    col=1,
)

fig.update_layout(
    height=760,
    title=f"Portfolio Reconstruction  |  {summary['period_start'][:10]} -> {summary['period_end'][:10]}",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.03),
    margin=dict(t=90),
)
fig.update_yaxes(title_text="USD", tickprefix="$", row=1, col=1)
fig.update_yaxes(title_text="USD", tickprefix="$", row=2, col=1)
fig.update_yaxes(title_text="%", ticksuffix="%", row=3, col=1)
fig.show()

## Step 5 — Per-Pair Cumulative PnL Contribution

In [5]:
df_events = pd.DataFrame(result["event_rows"])
df_events["time"] = pd.to_datetime(df_events["time"], format="%Y.%m.%d %H:%M:%S")

PAIR_COLORS = {
    "EURUSD": "#2196F3",
    "EURGBP": "#9C27B0",
    "GBPUSD": "#4CAF50",
    "USDCHF": "#FF9800",
}

# Cumulative PnL per pair
fig3 = go.Figure()
for pair in df_events["pair"].unique():
    sub = df_events[df_events["pair"] == pair].copy()
    sub = sub.sort_values("time")
    sub["cum_pnl"] = sub["scaled_net_profit"].cumsum()
    fig3.add_trace(go.Scatter(
        x=sub["time"],
        y=sub["cum_pnl"],
        name=pair,
        mode="lines",
        line=dict(color=PAIR_COLORS.get(pair), width=1.8),
    ))

fig3.update_layout(
    title="Cumulative Scaled PnL by Pair",
    height=420,
    hovermode="x unified",
    yaxis=dict(title="Cumulative PnL ($)", tickprefix="$"),
    legend=dict(orientation="h", y=1.04),
    margin=dict(t=80),
)
fig3.show()

# Pie chart — final contribution share
pairs_info = summary["pairs"]
fig4 = px.pie(
    names=list(pairs_info.keys()),
    values=[max(0, v["scaled_pnl_contribution"]) for v in pairs_info.values()],
    color=list(pairs_info.keys()),
    color_discrete_map=PAIR_COLORS,
    title="PnL Contribution Share",
    hole=0.4,
)
fig4.update_traces(textinfo="label+percent")
fig4.update_layout(height=380, margin=dict(t=60))
fig4.show()

## Step 7 - Scenario Configuration & Synthesis

**Edit the configuration table below to modify pair parameters. The analyzer will synthesize results by:**
- **Risk %**: Scale volumes/profits proportionally
- **Max Trades**: Limit to first N closed trades  
- **TP/Grid**: Displayed for reference (strategy parameters)

In [6]:
# Configuration baseline (from backtests)
import numpy as np

baseline_config = {
    'EURUSD':  {'risk': 3.3, 'tp': 15, 'grid': 450, 'max_trades': 6},
    'EURGBP':  {'risk': 1.2, 'tp': 5,  'grid': 450, 'max_trades': 4},
    'GBPUSD':  {'risk': 2.0, 'tp': 10, 'grid': 500, 'max_trades': 4},
    'USDCHF':  {'risk': 1.5, 'tp': 5,  'grid': 400, 'max_trades': 3},
}

pairs_by_name = {p.config.name: p for p in pairs_data}

# Display baseline configuration table
config_rows = []
for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    cfg = baseline_config[pair]
    config_rows.append({
        'Pair': pair,
        'Risk %': cfg['risk'],
        'TP': cfg['tp'],
        'Grid': cfg['grid'],
        'Max Trades': cfg['max_trades'],
    })
df_config = pd.DataFrame(config_rows)
display(df_config)

print("\n" + "=" * 80)
print("SCENARIO PARAMETERS - Edit below to change pair configuration")
print("=" * 80)

# ─ Scenario: Modify parameters here ───────────────────────────────────────────
scenario_config = {
    'EURUSD':  {'risk': 3.0, 'tp': 15, 'grid': 450, 'max_trades': 4},
    'EURGBP':  {'risk': 1.2, 'tp': 5,  'grid': 450, 'max_trades': 4},
    'GBPUSD':  {'risk': 1.8, 'tp': 10, 'grid': 450, 'max_trades': 4},
    'USDCHF':  {'risk': 1.5, 'tp': 5,  'grid': 400, 'max_trades': 3},
}
# ──────────────────────────────────────────────────────────────────────────────


def _open_count_at(count_rows, ts):
    if not count_rows:
        return 0
    idx = max(0, np.searchsorted([t for t, _ in count_rows], ts, side='right') - 1)
    return count_rows[idx][1]


def synthesize_pair_events(orig_pair, base_cfg, scen_cfg):
    """
    Build a synthesized pair from baseline deals/trades by:
    1) scaling included trades linearly by risk,
    2) enforcing scenario max-trades as a hard open-trade cap,
    3) scaling closed deal PnL for included legs,
    4) rebuilding pair floating curve so excluded legs do not contribute to floating PnL.
    """
    risk_factor = scen_cfg['risk'] / base_cfg['risk'] if base_cfg['risk'] > 0 else 1.0
    tp_factor = scen_cfg['tp'] / base_cfg['tp'] if base_cfg['tp'] > 0 else 1.0
    grid_factor = base_cfg['grid'] / scen_cfg['grid'] if scen_cfg['grid'] > 0 else 1.0
    pnl_factor = risk_factor * tp_factor * grid_factor

    base_max = int(base_cfg['max_trades']) if base_cfg['max_trades'] > 0 else 1
    scen_max = int(scen_cfg['max_trades']) if scen_cfg['max_trades'] > 0 else 1

    trades_sorted = sorted(orig_pair.trades, key=lambda t: t.time)

    modified_deals = []
    modified_trades = []

    # Maintain FIFO ledger so each out closes the corresponding earlier in.
    open_keep_flags = []
    kept_open = 0
    deal_idx = 0

    # Capture open-depth timeline for floating reconstruction.
    base_count_rows = []
    synth_count_rows = []
    base_open = 0

    for tr in trades_sorted:
        if tr.direction == 'in':
            base_open += 1
            keep_leg = kept_open < scen_max
            open_keep_flags.append(keep_leg)
            if keep_leg:
                kept_open += 1
                modified_trades.append(TradeEvent(
                    time=tr.time,
                    pair=tr.pair,
                    direction=tr.direction,
                    volume=tr.volume * risk_factor,
                    price=tr.price,
                ))

        elif tr.direction == 'out':
            base_open = max(0, base_open - 1)
            keep_leg = open_keep_flags.pop(0) if open_keep_flags else True

            deal = None
            if deal_idx < len(orig_pair.deals):
                deal = orig_pair.deals[deal_idx]
                deal_idx += 1

            if keep_leg:
                kept_open = max(0, kept_open - 1)
                modified_trades.append(TradeEvent(
                    time=tr.time,
                    pair=tr.pair,
                    direction=tr.direction,
                    volume=tr.volume * risk_factor,
                    price=tr.price,
                ))
                if deal is not None:
                    modified_deals.append(DealEvent(
                        time=deal.time,
                        pair=deal.pair,
                        net_profit=deal.net_profit * pnl_factor,
                        volume=deal.volume * risk_factor,
                    ))

        base_count_rows.append((tr.time, base_open))
        synth_count_rows.append((tr.time, kept_open))

    # Rebuild pair curve so floating excludes capped trades.
    # Floating scales by risk and by active-leg retention ratio (kept/base).
    synth_curve = []
    if orig_pair.curve:
        for cp in orig_pair.curve:
            base_open_now = _open_count_at(base_count_rows, cp.time)
            synth_open_now = _open_count_at(synth_count_rows, cp.time)
            if base_open_now > 0:
                floating_scale = risk_factor * (synth_open_now / base_open_now)
            else:
                floating_scale = 0.0

            base_floating = cp.equity - cp.balance
            synth_floating = base_floating * floating_scale
            cp_cls = type(cp)
            synth_curve.append(cp_cls(
                time=cp.time,
                balance=cp.balance,
                equity=cp.balance + synth_floating,
            ))

    return modified_deals, modified_trades, synth_curve, {
        'risk_factor': risk_factor,
        'tp_factor': tp_factor,
        'grid_factor': grid_factor,
        'pnl_factor': pnl_factor,
        'base_max_trades': base_max,
        'scenario_max_trades': scen_max,
        'deals_kept': len(modified_deals),
    }


# Build synthesized scenario pairs
scenario_pairs_data = []
scenario_factor_rows = []

for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    if pair not in pairs_by_name:
        continue

    orig_pair = pairs_by_name[pair]
    base_cfg = baseline_config[pair]
    scen_cfg = scenario_config[pair]

    synth_deals, synth_trades, synth_curve, factors = synthesize_pair_events(orig_pair, base_cfg, scen_cfg)

    synth_pair = PairData(
        config=PairConfig(
            name=pair,
            risk_percent=scen_cfg['risk'],
            base_lot=orig_pair.config.base_lot,
        ),
        deals=synth_deals,
        trades=synth_trades,
        curve=synth_curve,
        baseline_volume_median=orig_pair.baseline_volume_median,
    )
    scenario_pairs_data.append(synth_pair)

    scenario_factor_rows.append({
        'Pair': pair,
        'Risk x': round(factors['risk_factor'], 4),
        'TP x': round(factors['tp_factor'], 4),
        'Grid x': round(factors['grid_factor'], 4),
        'PnL x': round(factors['pnl_factor'], 4),
        'MaxTrades (B->S)': f"{factors['base_max_trades']}->{factors['scenario_max_trades']}",
        'Deals kept': factors['deals_kept'],
    })

print("\nSynthesis factors used:")
display(pd.DataFrame(scenario_factor_rows))

# Run synthesized scenario simulation
SCENARIO_BALANCE = 5000.0
SCENARIO_EXPONENT = 1.0
SCENARIO_MIN_SCALE = 0.1
SCENARIO_MAX_SCALE = 5.0

r2 = run_simulation(
    pairs_data=scenario_pairs_data,
    initial_balance=SCENARIO_BALANCE,
    scale_exponent=SCENARIO_EXPONENT,
    min_scale=SCENARIO_MIN_SCALE,
    max_scale=SCENARIO_MAX_SCALE,
    margin_requirements=margin_requirements,
)
s2 = apply_shared_dd(r2['summary'], r2['curve_rows'])
r2['summary'] = s2
scenario_avg_monthly_growth = average_monthly_balance_growth(r2['curve_rows'])

print(f"\nSynthesized scenario generated: {len(r2['event_rows'])} deals")
print(
    f"Return {s2['total_return_percent']:.2f}% | "
    f"MaxDD {s2['max_drawdown_percent']:.2f}% | "
    f"CAGR {s2['cagr_percent']:.2f}% | "
    f"Avg Monthly {scenario_avg_monthly_growth:.3f}%"
)

df_c2 = pd.DataFrame(r2['curve_rows'])
df_c2['time'] = pd.to_datetime(df_c2['time'], format='%Y.%m.%d %H:%M')
peak2 = df_c2['equity'].cummax()
dd2 = (peak2 - df_c2['equity']) / peak2 * 100

fig6 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Balance', 'Drawdown (%)'],
    vertical_spacing=0.06,
)

fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=df_c2['balance'],
    name='Scenario Balance', line=dict(color='#2196F3')
), row=1, col=1)
fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=df_c2['equity'],
    name='Scenario Equity', line=dict(color='#4CAF50', dash='dot')
), row=1, col=1)
fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=-dd2,
    fill='tozeroy', fillcolor='rgba(244,67,54,0.18)',
    line=dict(color='#F44336'), name='Scenario DD %'
), row=2, col=1)

fig6.update_layout(
    title=(
        f"Synthesized Scenario | Return {s2['total_return_percent']:.1f}% | "
        f"MaxDD {s2['max_drawdown_percent']:.1f}% | "
        f"CAGR {s2['cagr_percent']:.1f}% | "
        f"Avg Monthly {scenario_avg_monthly_growth:.3f}%"
    ),
    height=520,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.04),
    margin=dict(t=80),
)
fig6.update_yaxes(tickprefix='$', row=1, col=1)
fig6.update_yaxes(ticksuffix='%', row=2, col=1)
fig6.show()

,Pair,Risk %,TP,Grid,Max Trades
0,EURUSD,3.3,15,450,6
1,EURGBP,1.2,5,450,4
2,GBPUSD,2.0,10,500,4
3,USDCHF,1.5,5,400,3



SCENARIO PARAMETERS - Edit below to change pair configuration

Synthesis factors used:


,Pair,Risk x,TP x,Grid x,PnL x,MaxTrades (B->S),Deals kept
0,EURUSD,0.9091,1.0,1.0000,0.9091,6->4,334
1,EURGBP,1.0000,1.0,1.0000,1.0000,4->4,601
2,GBPUSD,0.9000,1.0,1.1111,1.0000,4->4,446
3,USDCHF,1.0000,1.0,1.0000,1.0000,3->3,1078



Synthesized scenario generated: 2459 deals
Return 2878.07% | MaxDD 103.73% | CAGR 169.58% | Avg Monthly 8.435%


In [7]:
# Find the time of maximum drawdown and show per-pair status at that moment
import numpy as np

# Calculate drawdown from df_curve
equity_vals = df_curve["equity"].values
peak_vals = pd.Series(equity_vals).cummax().values
dd_pct_vals = np.where(peak_vals > 0, (peak_vals - equity_vals) / peak_vals * 100.0, 0.0)

# Find max drawdown index and time
max_dd_idx = np.argmax(dd_pct_vals)
max_dd_time = df_curve.iloc[max_dd_idx]["time"]
max_dd_value = dd_pct_vals[max_dd_idx]
max_dd_abs = peak_vals[max_dd_idx] - equity_vals[max_dd_idx]

print(f"Maximum Drawdown Event:")
print(f"  Date: {max_dd_time}")
print(f"  DD %: {max_dd_value:.2f}%")
print(f"  DD $: ${max_dd_abs:,.2f}")
print(f"  Peak Equity: ${peak_vals[max_dd_idx]:,.2f}")
print(f"  Current Equity: ${equity_vals[max_dd_idx]:,.2f}")
print(f"  Balance: ${df_curve.iloc[max_dd_idx]['balance']:,.2f}")
print(f"  Floating PnL: ${df_curve.iloc[max_dd_idx]['floating_pnl']:,.2f}")

# Show per-pair status at that time
print(f"\n--- Per-Pair Floating PnL at {max_dd_time.strftime('%Y-%m-%d %H:%M')} ---")
for pair_data in pairs_data:
    # Interpolate floating PnL for this pair at the max DD time
    from mt5_portfolio_analyzer import _interpolate_floating
    float_pnl = _interpolate_floating(pair_data.curve, max_dd_time)
    
    # Get balance scale at that time
    idx = max(0, np.searchsorted(df_curve["time"].values, max_dd_time) - 1)
    balance_at_time = df_curve.iloc[idx]["balance"]
    bscale = balance_at_time / INITIAL_BALANCE if INITIAL_BALANCE > 0 else 1.0
    
    # Scaled floating PnL
    cfg = pair_data.config
    if cfg.base_lot is not None and pair_data.baseline_volume_median and pair_data.baseline_volume_median > 0:
        pm = cfg.base_lot / pair_data.baseline_volume_median
    else:
        pm = 1.0
    
    scaled_float = float_pnl * bscale * pm
    
    print(f"\n{pair_data.config.name}:")
    print(f"  Baseline floating PnL: ${float_pnl:,.2f}")
    print(f"  Balance scale: {bscale:.4f}")
    print(f"  Pair multiplier: {pm:.4f}")
    print(f"  Scaled floating PnL: ${scaled_float:,.2f}")

Maximum Drawdown Event:
  Date: 2023-10-03 18:48:00
  DD %: 134.83%
  DD $: $17,887.00
  Peak Equity: $13,266.31
  Current Equity: $-4,620.68
  Balance: $15,087.11
  Floating PnL: $-19,707.80

--- Per-Pair Floating PnL at 2023-10-03 18:48 ---

EURUSD:
  Baseline floating PnL: $-2,990.35
  Balance scale: 3.0174
  Pair multiplier: 1.0000
  Scaled floating PnL: $-9,023.15

EURGBP:
  Baseline floating PnL: $-144.40
  Balance scale: 3.0174
  Pair multiplier: 1.0000
  Scaled floating PnL: $-435.73

GBPUSD:
  Baseline floating PnL: $-2,117.94
  Balance scale: 3.0174
  Pair multiplier: 1.0000
  Scaled floating PnL: $-6,390.71

USDCHF:
  Baseline floating PnL: $-1,278.65
  Balance scale: 3.0174
  Pair multiplier: 1.0000
  Scaled floating PnL: $-3,858.21


## Step 7b - Scenario vs Baseline Comparison

In [8]:
# Display scenario configuration vs baseline
print("=" * 80)
print("SCENARIO CONFIGURATION (Baseline -> Synthesized Scenario)")
print("=" * 80)

scenario_rows = []
for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    b = baseline_config[pair]
    s = scenario_config[pair]
    scenario_rows.append({
        'Pair': pair,
        'Risk % (B->S)': f"{b['risk']} -> {s['risk']}",
        'TP (B->S)': f"{b['tp']} -> {s['tp']}",
        'Grid (B->S)': f"{b['grid']} -> {s['grid']}",
        'Max Trades (B->S)': f"{b['max_trades']} -> {s['max_trades']}",
    })

df_scenario_config = pd.DataFrame(scenario_rows)
display(df_scenario_config)

print("\n" + "=" * 80)
print("SYNTHESIS SUMMARY")
print("=" * 80)
print(f"Baseline deals: {len(result['event_rows'])}")
print(f"Synthesized scenario deals: {len(r2['event_rows'])}")

baseline_monthly_growth = average_monthly_balance_growth(result['curve_rows'])
scenario_monthly_growth = average_monthly_balance_growth(r2['curve_rows'])


def calc_min_margin(df_in):
    """Minimum MT4/MT5 margin level %: equity / used_margin * 100."""
    if df_in is None:
        return None

    if 'margin_level_percent' in df_in.columns:
        margin = pd.to_numeric(df_in['margin_level_percent'], errors='coerce')
        margin = margin.replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    if 'used_margin' in df_in.columns and 'equity' in df_in.columns:
        used = pd.to_numeric(df_in['used_margin'], errors='coerce')
        eq = pd.to_numeric(df_in['equity'], errors='coerce')
        margin = (eq / used.replace(0, np.nan) * 100).replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    return None


baseline_margin = calc_min_margin(pd.DataFrame(result['curve_rows']))
scenario_margin = calc_min_margin(pd.DataFrame(r2['curve_rows']))

# Optional validation against actual proposed backtest files
proposed_validation = None
proposed_validation_monthly = None
proposed_validation_margin = None
PROPOSED_DIR = os.path.join(REPO, 'data', 'proposed')
if os.path.exists(PROPOSED_DIR) and len(os.listdir(PROPOSED_DIR)) > 0:
    discovered_proposed = discover_files(PROPOSED_DIR)
    pairs_data_proposed = []
    for pair, files in discovered_proposed.items():
        xlsx = files.get('xlsx')
        csv = files.get('csv')
        if not xlsx:
            continue
        pairs_data_proposed.append(load_pair(
            name=pair,
            risk_percent=_PAIR_RISK.get(pair, 0.0),
            base_lot=_PAIR_BASE_LOT.get(pair),
            xlsx_path=xlsx,
            csv_path=csv,
        ))

    if pairs_data_proposed:
        proposed_result = run_simulation(
            pairs_data=pairs_data_proposed,
            initial_balance=INITIAL_BALANCE,
            scale_exponent=SCALE_EXPONENT,
            min_scale=MIN_SCALE,
            max_scale=MAX_SCALE,
            margin_requirements=margin_requirements,
        )
        proposed_validation = apply_shared_dd(proposed_result['summary'], proposed_result['curve_rows'])
        proposed_validation_monthly = average_monthly_balance_growth(proposed_result['curve_rows'])
        proposed_validation_margin = calc_min_margin(pd.DataFrame(proposed_result['curve_rows']))
        print("Validation using actual proposed backtests loaded.")

# Compare baseline vs synthesized, and optionally vs actual proposed
comparison_data = {
    'Metric': [
        'Final Balance',
        'Total Return (%)',
        'CAGR (%)',
        'Avg Monthly Balance Growth (%)',
        'Min Margin Level (%)',
        'Max Drawdown ($)',
        'Max Drawdown (%)',
        'Final Equity',
    ],
    'Baseline': [
        f"${summary['final_balance']:,.2f}",
        f"{summary['total_return_percent']:.2f}%",
        f"{summary['cagr_percent']:.2f}%",
        f"{baseline_monthly_growth:.3f}%" if baseline_monthly_growth is not None else 'N/A',
        f"{baseline_margin:.2f}%" if baseline_margin is not None else 'N/A',
        f"${summary['max_drawdown_abs']:,.2f}",
        f"{summary['max_drawdown_percent']:.2f}%",
        f"${summary['final_equity']:,.2f}",
    ],
    'Synthesized Scenario': [
        f"${s2['final_balance']:,.2f}",
        f"{s2['total_return_percent']:.2f}%",
        f"{s2['cagr_percent']:.2f}%",
        f"{scenario_monthly_growth:.3f}%" if scenario_monthly_growth is not None else 'N/A',
        f"{scenario_margin:.2f}%" if scenario_margin is not None else 'N/A',
        f"${s2['max_drawdown_abs']:,.2f}",
        f"{s2['max_drawdown_percent']:.2f}%",
        f"${s2['final_equity']:,.2f}",
    ],
    'Delta vs Baseline': [
        f"${s2['final_balance'] - summary['final_balance']:+,.2f}",
        f"{s2['total_return_percent'] - summary['total_return_percent']:+.2f}pp",
        f"{s2['cagr_percent'] - summary['cagr_percent']:+.2f}pp",
        (
            f"{scenario_monthly_growth - baseline_monthly_growth:+.3f}pp"
            if (scenario_monthly_growth is not None and baseline_monthly_growth is not None)
            else 'N/A'
        ),
        (
            f"{scenario_margin - baseline_margin:+.2f}pp"
            if (scenario_margin is not None and baseline_margin is not None)
            else 'N/A'
        ),
        f"${s2['max_drawdown_abs'] - summary['max_drawdown_abs']:+,.2f}",
        f"{s2['max_drawdown_percent'] - summary['max_drawdown_percent']:+.2f}pp",
        f"${s2['final_equity'] - summary['final_equity']:+,.2f}",
    ],
}

if proposed_validation is not None:
    comparison_data['Actual Proposed'] = [
        f"${proposed_validation['final_balance']:,.2f}",
        f"{proposed_validation['total_return_percent']:.2f}%",
        f"{proposed_validation['cagr_percent']:.2f}%",
        f"{proposed_validation_monthly:.3f}%" if proposed_validation_monthly is not None else 'N/A',
        f"{proposed_validation_margin:.2f}%" if proposed_validation_margin is not None else 'N/A',
        f"${proposed_validation['max_drawdown_abs']:,.2f}",
        f"{proposed_validation['max_drawdown_percent']:.2f}%",
        f"${proposed_validation['final_equity']:,.2f}",
    ]
    comparison_data['Synth Error vs Actual'] = [
        f"${s2['final_balance'] - proposed_validation['final_balance']:+,.2f}",
        f"{s2['total_return_percent'] - proposed_validation['total_return_percent']:+.2f}pp",
        f"{s2['cagr_percent'] - proposed_validation['cagr_percent']:+.2f}pp",
        (
            f"{scenario_monthly_growth - proposed_validation_monthly:+.3f}pp"
            if (scenario_monthly_growth is not None and proposed_validation_monthly is not None)
            else 'N/A'
        ),
        (
            f"{scenario_margin - proposed_validation_margin:+.2f}pp"
            if (scenario_margin is not None and proposed_validation_margin is not None)
            else 'N/A'
        ),
        f"${s2['max_drawdown_abs'] - proposed_validation['max_drawdown_abs']:+,.2f}",
        f"{s2['max_drawdown_percent'] - proposed_validation['max_drawdown_percent']:+.2f}pp",
        f"${s2['final_equity'] - proposed_validation['final_equity']:+,.2f}",
    ]

df_synth_comp = pd.DataFrame(comparison_data)
display(df_synth_comp)

SCENARIO CONFIGURATION (Baseline -> Synthesized Scenario)


,Pair,Risk % (B->S),TP (B->S),Grid (B->S),Max Trades (B->S)
0,EURUSD,3.3 -> 3.0,15 -> 15,450 -> 450,6 -> 4
1,EURGBP,1.2 -> 1.2,5 -> 5,450 -> 450,4 -> 4
2,GBPUSD,2.0 -> 1.8,10 -> 10,500 -> 450,4 -> 4
3,USDCHF,1.5 -> 1.5,5 -> 5,400 -> 400,3 -> 3



SYNTHESIS SUMMARY
Baseline deals: 2461
Synthesized scenario deals: 2459
Validation using actual proposed backtests loaded.


,Metric,Baseline,Synthesized Scenario,Delta vs Baseline,Actual Proposed,Synth Error vs Actual
0,Final Balance,"$154,949.70","$148,903.66","$-6,046.04","$150,095.81","$-1,192.15"
1,Total Return (%),2998.99%,2878.07%,-120.92pp,2901.92%,-23.84pp
2,CAGR (%),172.73%,169.58%,-3.15pp,170.21%,-0.63pp
3,Avg Monthly Balance Growth (%),8.536%,8.435%,-0.101pp,8.449%,-0.014pp
4,Min Margin Level (%),-20.94%,-2.77%,+18.17pp,-15.41%,+12.64pp
5,Max Drawdown ($),"$17,887.00","$13,242.93","$-4,644.07","$15,352.85","$-2,109.92"
6,Max Drawdown (%),134.83%,103.73%,-31.10pp,122.09%,-18.36pp
7,Final Equity,"$154,949.70","$148,903.66","$-6,046.04","$150,095.81","$-1,192.15"


In [13]:
# Step 7b.1 - EURUSD Risk Sensitivity Sanity Check (isolated vs mixed)
# This isolates EURUSD risk-only impact so it is not conflated with max_trades/other-pair changes.

def _run_scenario_with_config(cfg_map):
    local_pairs = []
    for _pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
        if _pair not in pairs_by_name:
            continue
        _orig = pairs_by_name[_pair]
        _base = baseline_config[_pair]
        _scen = cfg_map[_pair]
        _deals, _trades, _curve, _factors = synthesize_pair_events(_orig, _base, _scen)
        local_pairs.append(PairData(
            config=PairConfig(name=_pair, risk_percent=_scen['risk'], base_lot=_orig.config.base_lot),
            deals=_deals,
            trades=_trades,
            curve=_curve,
            baseline_volume_median=_orig.baseline_volume_median,
        ))

    _r = run_simulation(
        pairs_data=local_pairs,
        initial_balance=SCENARIO_BALANCE,
        scale_exponent=SCENARIO_EXPONENT,
        min_scale=SCENARIO_MIN_SCALE,
        max_scale=SCENARIO_MAX_SCALE,
        margin_requirements=margin_requirements,
    )
    _s = apply_shared_dd(_r['summary'], _r['curve_rows'])
    _r['summary'] = _s
    _monthly = average_monthly_balance_growth(_r['curve_rows'])
    return _r, _s, _monthly


# Mixed scenario already computed in r2/s2/scenario_monthly_growth
base_eu = summary['pairs']['EURUSD']['scaled_pnl_contribution']
mixed_eu = s2['pairs']['EURUSD']['scaled_pnl_contribution']

# Isolated scenario: only EURUSD risk 3.3 -> 3.0, keep all other knobs at baseline.
isolated_cfg = {k: dict(v) for k, v in baseline_config.items()}
isolated_cfg['EURUSD']['risk'] = 3.0

r_iso, s_iso, monthly_iso = _run_scenario_with_config(isolated_cfg)
iso_eu = s_iso['pairs']['EURUSD']['scaled_pnl_contribution']

risk_linear_factor = isolated_cfg['EURUSD']['risk'] / baseline_config['EURUSD']['risk']

sanity_rows = [
    {
        'Case': 'EURUSD Contribution - Baseline',
        'Value': f"${base_eu:,.2f}",
    },
    {
        'Case': 'EURUSD Contribution - Mixed Scenario (Step 7)',
        'Value': f"${mixed_eu:,.2f} ({(mixed_eu / base_eu - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'EURUSD Contribution - Isolated Risk-Only (3.3 -> 3.0)',
        'Value': f"${iso_eu:,.2f} ({(iso_eu / base_eu - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'Expected Linear Risk Factor (3.0/3.3)',
        'Value': f"{risk_linear_factor:.4f} ({(risk_linear_factor - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'Portfolio Avg Monthly - Baseline',
        'Value': f"{baseline_monthly_growth:.3f}%" if baseline_monthly_growth is not None else 'N/A',
    },
    {
        'Case': 'Portfolio Avg Monthly - Mixed Scenario',
        'Value': f"{scenario_monthly_growth:.3f}% ({scenario_monthly_growth - baseline_monthly_growth:+.3f}pp)"
                 if (scenario_monthly_growth is not None and baseline_monthly_growth is not None) else 'N/A',
    },
    {
        'Case': 'Portfolio Avg Monthly - Isolated EURUSD Risk-Only',
        'Value': f"{monthly_iso:.3f}% ({monthly_iso - baseline_monthly_growth:+.3f}pp)"
                 if (monthly_iso is not None and baseline_monthly_growth is not None) else 'N/A',
    },
]

display(pd.DataFrame(sanity_rows))

,Case,Value
0,EURUSD Contribution - Baseline,"$86,010.61"
1,EURUSD Contribution - Mixed Scenario (Step 7),"$79,895.95 (-7.11%)"
2,EURUSD Contribution - Isolated Risk-Only (3.3 ...,"$77,822.89 (-9.52%)"
3,Expected Linear Risk Factor (3.0/3.3),0.9091 (-9.09%)
4,Portfolio Avg Monthly - Baseline,8.536%
5,Portfolio Avg Monthly - Mixed Scenario,8.435% (-0.101pp)
6,Portfolio Avg Monthly - Isolated EURUSD Risk-Only,8.384% (-0.152pp)


In [9]:
# Step 7c - Max Drawdown Forensics (Baseline vs Synthesized)
import numpy as np


def _pair_multiplier(pair_data):
    cfg = pair_data.config
    if cfg.base_lot is not None and pair_data.baseline_volume_median and pair_data.baseline_volume_median > 0:
        return cfg.base_lot / pair_data.baseline_volume_median
    return 1.0


def _scenario_dd_context(curve_rows, pair_data_list, label, initial_balance):
    """Return max-DD context and per-pair floating contribution at DD timestamp."""
    df = pd.DataFrame(curve_rows).copy()
    df['time'] = pd.to_datetime(df['time'], format='%Y.%m.%d %H:%M')

    equity_vals = df['equity'].to_numpy()
    peak_vals = pd.Series(equity_vals).cummax().to_numpy()
    dd_abs_vals = peak_vals - equity_vals
    dd_pct_vals = np.where(peak_vals > 0, dd_abs_vals / peak_vals * 100.0, 0.0)

    dd_idx = int(np.argmax(dd_pct_vals))
    dd_time = df.iloc[dd_idx]['time']
    dd_pct = float(dd_pct_vals[dd_idx])
    dd_abs = float(dd_abs_vals[dd_idx])
    peak_eq = float(peak_vals[dd_idx])
    cur_eq = float(equity_vals[dd_idx])

    bal_idx = max(0, np.searchsorted(df['time'].to_numpy(), dd_time) - 1)
    bal_at_time = float(df.iloc[bal_idx]['balance'])
    bscale = bal_at_time / initial_balance if initial_balance > 0 else 1.0

    rows = []
    total_float = 0.0
    for pair_data in pair_data_list:
        pm = _pair_multiplier(pair_data)
        base_float = _interpolate_floating(pair_data.curve, dd_time)
        scaled_float = base_float * bscale * pm
        total_float += scaled_float
        rows.append({
            'Scenario': label,
            'DD Timestamp': dd_time.strftime('%Y-%m-%d %H:%M'),
            'Pair': pair_data.config.name,
            'Base Floating PnL': base_float,
            'Balance Scale': bscale,
            'Pair Multiplier': pm,
            'Scaled Floating PnL': scaled_float,
        })

    summary = {
        'Scenario': label,
        'DD Timestamp': dd_time.strftime('%Y-%m-%d %H:%M'),
        'Max DD %': dd_pct,
        'Max DD $': dd_abs,
        'Peak Equity': peak_eq,
        'Equity @ DD': cur_eq,
        'Balance @ DD': bal_at_time,
        'Total Floating @ DD': total_float,
    }
    return summary, rows


baseline_dd_summary, baseline_dd_rows = _scenario_dd_context(
    result['curve_rows'],
    pairs_data,
    'Baseline',
    INITIAL_BALANCE,
)

synth_dd_summary, synth_dd_rows = _scenario_dd_context(
    r2['curve_rows'],
    scenario_pairs_data,
    'Synthesized',
    INITIAL_BALANCE,
)

print('Max Drawdown Context (Portfolio-Level)')
df_dd_summary = pd.DataFrame([baseline_dd_summary, synth_dd_summary])
for col in ['Max DD %', 'Max DD $', 'Peak Equity', 'Equity @ DD', 'Balance @ DD', 'Total Floating @ DD']:
    df_dd_summary[col] = pd.to_numeric(df_dd_summary[col], errors='coerce')
display(df_dd_summary)

print('\nPer-Pair Floating Contribution At Each Scenario\'s DD Timestamp')
df_dd_pairs = pd.DataFrame(baseline_dd_rows + synth_dd_rows)
for col in ['Base Floating PnL', 'Balance Scale', 'Pair Multiplier', 'Scaled Floating PnL']:
    df_dd_pairs[col] = pd.to_numeric(df_dd_pairs[col], errors='coerce')

display(
    df_dd_pairs.sort_values(['Scenario', 'Pair'])
)

print('\nPer-Pair Delta In Scaled Floating PnL (Synth - Baseline)')
base_p = pd.DataFrame(baseline_dd_rows)[['Pair', 'Scaled Floating PnL']].rename(columns={'Scaled Floating PnL': 'Baseline Scaled Floating'})
syn_p = pd.DataFrame(synth_dd_rows)[['Pair', 'Scaled Floating PnL']].rename(columns={'Scaled Floating PnL': 'Synth Scaled Floating'})
delta_p = base_p.merge(syn_p, on='Pair', how='outer')
delta_p['Delta (Synth - Baseline)'] = delta_p['Synth Scaled Floating'] - delta_p['Baseline Scaled Floating']
display(delta_p.sort_values('Pair'))

Max Drawdown Context (Portfolio-Level)


,Scenario,DD Timestamp,Max DD %,Max DD $,Peak Equity,Equity @ DD,Balance @ DD,Total Floating @ DD
0,Baseline,2023-10-03 18:48,134.830211,17886.9976,13266.3128,-4620.6848,15087.1127,-19707.797456
1,Synthesized,2023-10-04 10:02,103.729027,13242.9313,12766.8520,-476.0793,14530.3095,-15006.388775



Per-Pair Floating Contribution At Each Scenario's DD Timestamp


,Scenario,DD Timestamp,Pair,Base Floating PnL,Balance Scale,Pair Multiplier,Scaled Floating PnL
1,Baseline,2023-10-03 18:48,EURGBP,-144.403949,3.017423,1.0,-435.727729
0,Baseline,2023-10-03 18:48,EURUSD,-2990.350000,3.017423,1.0,-9023.149492
2,Baseline,2023-10-03 18:48,GBPUSD,-2117.936043,3.017423,1.0,-6390.707955
3,Baseline,2023-10-03 18:48,USDCHF,-1278.645012,3.017423,1.0,-3858.212279
5,Synthesized,2023-10-04 10:02,EURGBP,-146.889988,2.906062,1.0,-426.871399
4,Synthesized,2023-10-04 10:02,EURUSD,-1758.510232,2.906062,1.0,-5110.339587
6,Synthesized,2023-10-04 10:02,GBPUSD,-1997.919000,2.906062,1.0,-5806.076285
7,Synthesized,2023-10-04 10:02,USDCHF,-1260.503606,2.906062,1.0,-3663.101504



Per-Pair Delta In Scaled Floating PnL (Synth - Baseline)


,Pair,Baseline Scaled Floating,Synth Scaled Floating,Delta (Synth - Baseline)
0,EURGBP,-435.727729,-426.871399,8.856331
1,EURUSD,-9023.149492,-5110.339587,3912.809905
2,GBPUSD,-6390.707955,-5806.076285,584.631670
3,USDCHF,-3858.212279,-3663.101504,195.110775


## Step 8 — Baseline vs Proposed Comparison

Compare key metrics between two scenarios: baseline and proposed strategy parameters.
Add backtest files to `data/proposed/` folder to enable this comparison.


In [10]:
import os
import numpy as np

# Load baseline (already loaded above as result/summary)
baseline_summary = apply_shared_dd(summary, result['curve_rows'])
baseline_result = result
baseline_df_curve = pd.DataFrame(result['curve_rows'])
baseline_df_curve['time'] = pd.to_datetime(baseline_df_curve['time'], format='%Y.%m.%d %H:%M')

# Try to load proposed scenario
PROPOSED_DIR = os.path.join(REPO, 'data', 'proposed')
proposed_summary = None
proposed_result = None
proposed_df_curve = None

if os.path.exists(PROPOSED_DIR) and len(os.listdir(PROPOSED_DIR)) > 0:
    print('Loading proposed scenario...')
    discovered_proposed = discover_files(PROPOSED_DIR)
    pairs_data_proposed = []

    for pair, files in discovered_proposed.items():
        xlsx = files.get('xlsx')
        csv = files.get('csv')
        if not xlsx:
            continue
        pairs_data_proposed.append(load_pair(
            name=pair,
            risk_percent=_PAIR_RISK.get(pair, 0.0),
            base_lot=_PAIR_BASE_LOT.get(pair),
            xlsx_path=xlsx,
            csv_path=csv,
        ))

    if pairs_data_proposed:
        proposed_result = run_simulation(
            pairs_data=pairs_data_proposed,
            initial_balance=INITIAL_BALANCE,
            scale_exponent=SCALE_EXPONENT,
            min_scale=MIN_SCALE,
            max_scale=MAX_SCALE,
            margin_requirements=margin_requirements,
        )
        proposed_summary = apply_shared_dd(proposed_result['summary'], proposed_result['curve_rows'])
        proposed_df_curve = pd.DataFrame(proposed_result['curve_rows'])
        proposed_df_curve['time'] = pd.to_datetime(proposed_df_curve['time'], format='%Y.%m.%d %H:%M')
        print(f"Proposed loaded: {proposed_summary['total_deals']} deals\n")


def calc_monthly_growth(df_in):
    """Average month-over-month balance growth %"""
    if df_in is None or 'balance' not in df_in.columns or 'time' not in df_in.columns:
        return None
    df = df_in.copy()
    df['ym'] = df['time'].dt.to_period('M')
    monthly_bal = df.groupby('ym')['balance'].last()
    if len(monthly_bal) < 2:
        return None
    returns = monthly_bal.pct_change().dropna()
    return returns.mean() * 100 if len(returns) > 0 else None


def calc_min_margin(df_in):
    """Minimum MT4/MT5 margin level %: equity / used_margin * 100."""
    if df_in is None:
        return None

    if 'margin_level_percent' in df_in.columns:
        margin = pd.to_numeric(df_in['margin_level_percent'], errors='coerce')
        margin = margin.replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    if 'used_margin' in df_in.columns and 'equity' in df_in.columns:
        used = pd.to_numeric(df_in['used_margin'], errors='coerce')
        eq = pd.to_numeric(df_in['equity'], errors='coerce')
        margin = (eq / used.replace(0, np.nan) * 100).replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    return None


baseline_monthly = calc_monthly_growth(baseline_df_curve)
baseline_margin = calc_min_margin(baseline_df_curve)
proposed_monthly = calc_monthly_growth(proposed_df_curve)
proposed_margin = calc_min_margin(proposed_df_curve)

comp = {
    'Metric': [
        'Initial Balance', 'Final Balance', 'Total Return %', 'CAGR %',
        'Max Drawdown %', 'Avg Monthly Growth %', 'Min Margin Level %',
        'Total Deals', 'Period Start'
    ],
    'Baseline': [
        f"${baseline_summary['initial_balance']:,.0f}",
        f"${baseline_summary['final_balance']:,.0f}",
        f"{baseline_summary['total_return_percent']:.2f}%",
        f"{baseline_summary['cagr_percent']:.2f}%",
        f"{baseline_summary['max_drawdown_percent']:.2f}%",
        f"{baseline_monthly:.3f}%" if baseline_monthly is not None else 'N/A',
        f"{baseline_margin:.2f}%" if baseline_margin is not None else 'N/A',
        f"{baseline_summary['total_deals']}",
        baseline_summary['period_start'][:10],
    ]
}

if proposed_summary is not None:
    comp['Proposed'] = [
        f"${proposed_summary['initial_balance']:,.0f}",
        f"${proposed_summary['final_balance']:,.0f}",
        f"{proposed_summary['total_return_percent']:.2f}%",
        f"{proposed_summary['cagr_percent']:.2f}%",
        f"{proposed_summary['max_drawdown_percent']:.2f}%",
        f"{proposed_monthly:.3f}%" if proposed_monthly is not None else 'N/A',
        f"{proposed_margin:.2f}%" if proposed_margin is not None else 'N/A',
        f"{proposed_summary['total_deals']}",
        proposed_summary['period_start'][:10],
    ]
    comp['Difference'] = [
        '',
        f"{((proposed_summary['final_balance'] / baseline_summary['final_balance']) - 1) * 100:+.1f}%",
        f"{proposed_summary['total_return_percent'] - baseline_summary['total_return_percent']:+.2f}pp",
        f"{proposed_summary['cagr_percent'] - baseline_summary['cagr_percent']:+.2f}pp",
        f"{proposed_summary['max_drawdown_percent'] - baseline_summary['max_drawdown_percent']:+.2f}pp",
        f"{proposed_monthly - baseline_monthly:+.3f}pp" if (proposed_monthly is not None and baseline_monthly is not None) else 'N/A',
        f"{proposed_margin - baseline_margin:+.2f}pp" if (proposed_margin is not None and baseline_margin is not None) else 'N/A',
        '',
        '',
    ]

df_comp = pd.DataFrame(comp)
display(df_comp.style.hide(axis='index').set_caption('Strategy Comparison: Baseline vs Proposed'))

Loading proposed scenario...
Proposed loaded: 2453 deals



Metric,Baseline,Proposed,Difference
Initial Balance,"$5,000","$5,000",
Final Balance,"$154,950","$150,096",-3.1%
Total Return %,2998.99%,2901.92%,-97.08pp
CAGR %,172.73%,170.21%,-2.52pp
Max Drawdown %,134.83%,122.09%,-12.74pp
Avg Monthly Growth %,8.536%,8.449%,-0.087pp
Min Margin Level %,-20.94%,-15.41%,+5.53pp
Total Deals,2461,2453,
Period Start,2023.01.02,2023.01.02,


In [11]:
# Quick verification: actual unclamped drawdown and minimum equity
baseline_dd = baseline_summary['max_drawdown_percent']
proposed_dd = proposed_summary['max_drawdown_percent'] if proposed_summary is not None else None

print(f"Baseline Max Drawdown (unclamped): {baseline_dd:.4f}%")
if proposed_dd is not None:
    print(f"Proposed Max Drawdown (unclamped): {proposed_dd:.4f}%")

baseline_raw_equity = pd.DataFrame(baseline_result['curve_rows'])['equity']
proposed_raw_equity = pd.DataFrame(proposed_result['curve_rows'])['equity'] if proposed_result is not None else None

print(f"\nBaseline minimum equity: ${baseline_raw_equity.min():,.2f}")
if proposed_raw_equity is not None:
    print(f"Proposed minimum equity: ${proposed_raw_equity.min():,.2f}")

Baseline Max Drawdown (unclamped): 134.8302%
Proposed Max Drawdown (unclamped): 122.0854%

Baseline minimum equity: $-4,620.68
Proposed minimum equity: $-2,777.35
